# ROR raw EDA

## Things i need to find out

* Geolocation, country, citz
 
## Sanitization

We can use Python UDF's for that

```
con.create_function("parse_names", parse_names_and_identifiers)
con.execute("""
    UPDATE organization 
    SET legalName = parse_names(legalName)
""")
```

In [1]:
import sys
sys.path.insert(0, '/home/lu72hip/DIGICHer/dh_pipeline/src')

import pandas as pd
from pathlib import Path
from lib.database.duck.create_connection import create_duck_connection
from utils.config.config_loader import get_query_config

config = get_query_config()['ror_dump']
DB = Path(config['path_duck'])

con = create_duck_connection(str(DB))
con.execute("SET memory_limit='32GB'")
con.execute("SET threads=64")
print(f'Connected to {DB}')

pd.set_option("display.max_rows", 50) # Show more rows  # or None for unlimited
pd.set_option("display.max_columns", None) # Show more columns  # None = show all
pd.set_option("display.max_colwidth", None) # Don't truncate column content  # None = full content
pd.set_option("display.width", None) # Wider display  # Auto-detect terminal width

/home/lu72hip/DIGICHer/dh_pipeline/config/config_queries.json {'arxiv': {'checkpoint': 'submittedDateTo', 'queries': [{'query': '(all:cultural+OR+all:heritage+OR+all:digital+OR+all:humanities)', 'download_attachments': True, 'checkpoint_start': '1990-01-01-00-00', 'checkpoint_range': '6'}, {'query': 'all:computing+AND+(all:humanities+OR+all:heritage)', 'download_attachments': True, 'checkpoint_start': '1990-01-01-00-00', 'checkpoint_range': '6'}]}, 'cordis': {'checkpoint': 'startDate', 'queries': [{'query': 'cultural OR heritage OR digital OR humanities', 'download_attachments': True, 'checkpoint_start': '1985-01-01', 'checkpoint_range': '1'}, {'query': "contenttype='project' AND '*'", 'download_attachments': False, 'checkpoint_start': '1985-01-01', 'checkpoint_range': '1', 'path_duck': '/vast/lu72hip/data/duckdb/sources/cordis_raw.duckdb'}]}, 'coreac': {'checkpoint': 'publishedDate', 'queries': [{'query': '((computing AND cultural) OR (computing AND heritage))', 'download_attachments'

## Stats

In [2]:
q = """ 
SELECT
    table_name,
    estimated_size as cnt
FROM duckdb_tables()
WHERE schema_name = 'main'
ORDER BY estimated_size DESC
"""
con.execute(q).df()

,table_name,cnt
0,organizations,122388


## Entity Organization

In [2]:
### Sample Columns ###
q = """ 
SELECT * FROM organizations
LIMIT 1;
"""
display(con.execute(q).df())

,locations,established,external_ids,id,domains,links,names,relationships,status,types,admin
0,"[{'geonames_id': 2158177, 'geonames_details': {'continent_code': 'OC', 'continent_name': 'Oceania', 'country_code': 'AU', 'country_name': 'Australia', 'country_subdivision_code': 'VIC', 'country_subdivision_name': 'Victoria', 'lat': -37.814, 'lng': 144.96332, 'name': 'Melbourne'}}]",1887,"[{'type': 'fundref', 'all': ['501100001780', '100008690', '100010552'], 'preferred': '501100001780'}, {'type': 'grid', 'all': ['grid.1017.7'], 'preferred': 'grid.1017.7'}, {'type': 'isni', 'all': ['0000 0001 2163 3550'], 'preferred': None}, {'type': 'wikidata', 'all': ['Q1057890'], 'preferred': None}]",https://ror.org/04ttjf776,[],"[{'type': 'website', 'value': 'https://www.rmit.edu.au/'}, {'type': 'wikipedia', 'value': 'http://en.wikipedia.org/wiki/RMIT_University'}]","[{'value': 'RMIT', 'types': ['acronym'], 'lang': None}, {'value': 'RMIT University', 'types': ['ror_display', 'label'], 'lang': 'en'}, {'value': 'Royal Melbourne Institute of Technology University', 'types': ['alias'], 'lang': 'en'}]","[{'type': 'child', 'label': 'ARC Centre of Excellence for Automated Decision-Making and Society', 'id': 'https://ror.org/039p7nx39'}, {'type': 'child', 'label': 'RMIT Europe', 'id': 'https://ror.org/03m3ca021'}, {'type': 'child', 'label': 'RMIT Vietnam', 'id': 'https://ror.org/004axh929'}, {'type': 'related', 'label': 'Austin Hospital', 'id': 'https://ror.org/010mv7n52'}]",active,"[education, funder]","{'created': {'date': 2018-11-14, 'schema_version': '1.0'}, 'last_modified': {'date': 2024-12-11, 'schema_version': '2.1'}}"


In [11]:
# Seach FSU Jena

q = """ 
SELECT * 
FROM organizations
WHERE EXISTS (
    FROM UNNEST(names) AS n
    WHERE n.unnest.value ilike '%jena%'
    AND n.unnest.value ilike '%Schiller%'
)
"""
display(con.execute(q).df())

,locations,established,external_ids,id,domains,links,names,relationships,status,types,admin
0,"[{'geonames_id': 2895044, 'geonames_details': {'continent_code': 'EU', 'continent_name': 'Europe', 'country_code': 'DE', 'country_name': 'Germany', 'country_subdivision_code': 'TH', 'country_subdivision_name': 'Thuringia', 'lat': 50.92878, 'lng': 11.5899, 'name': 'Jena'}}]",1558,"[{'type': 'fundref', 'all': ['100012957'], 'preferred': '100012957'}, {'type': 'grid', 'all': ['grid.9613.d'], 'preferred': 'grid.9613.d'}, {'type': 'isni', 'all': ['0000 0001 1939 2794'], 'preferred': None}, {'type': 'wikidata', 'all': ['Q154561'], 'preferred': None}]",https://ror.org/05qpz1x62,[],"[{'type': 'website', 'value': 'https://www.uni-jena.de/en'}, {'type': 'wikipedia', 'value': 'http://en.wikipedia.org/wiki/University_of_Jena'}]","[{'value': 'FSU', 'types': ['acronym'], 'lang': None}, {'value': 'Friedrich Schiller University Jena', 'types': ['ror_display', 'label'], 'lang': 'en'}, {'value': 'Friedrich-Schiller-Universität Jena', 'types': ['label'], 'lang': 'de'}]","[{'type': 'child', 'label': 'Thüringer Universitäts- und Landesbibliothek', 'id': 'https://ror.org/0493yt138'}, {'type': 'related', 'label': 'Jena University Hospital', 'id': 'https://ror.org/035rzkx15'}, {'type': 'related', 'label': 'Zentralklinik Bad Berka', 'id': 'https://ror.org/00zfe1b87'}, {'type': 'related', 'label': 'German Centre for Integrative Biodiversity Research', 'id': 'https://ror.org/01jty7g66'}]",active,"[education, funder]","{'created': {'date': 2018-11-14, 'schema_version': '1.0'}, 'last_modified': {'date': 2025-01-22, 'schema_version': '2.1'}}"
